# Homework 1

#### EE-556 Mathematics of Data - Fall 2026


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Stefano Negro**: || Person 1 **424721**:


Person 2 **Vittorio Garavelli**: || Person 2 **427761**:


Person 3 **Emanuele Albert**: || Person 3 **SCIPER**:


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

*Describe your use of AI tools here.*


## 1. Multiclass Softmax Logistic Regression - 30 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (2 points) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



Given the assumption that the samples are independent and identically distributed (i.i.d.), we can convert the joint probability into a product of individual probabilities, and then use logarithm properties to expand the expression:

$$
\begin{aligned}
f(\mathbf{X}) &= -\log \mathbb{P}(b_1, \dots, b_n \mid \mathbf{a}_1, \dots, \mathbf{a}_n) \\
&= -\log \left( \prod_{i=1}^n \mathbb{P}(b_i \mid \mathbf{a}_i) \right) \\
&= -\sum_{i=1}^n \log \mathbb{P}(b_i \mid \mathbf{a}_i) 
\end{aligned}
$$

Next, substitute the definition of the softmax model for each sample's true class $b_i$, and use the quotient rule of logarithms ($\log(x/y) = \log(x) - \log(y)$):

$$
\begin{aligned}
f(\mathbf{X}) &= -\sum_{i=1}^n \log \left( \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \right) \\
&= -\sum_{i=1}^n \left[ \log \left( \exp(\mathbf{a}_i^\top \mathbf{x}_{b_i}) \right) - \log \left( \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \right]
\end{aligned}
$$

Since the natural logarithm and exponential function are inverses, the first term simplifies to the exponent. Finally, distributing the negative sign yields the target equation:

$$
\begin{aligned}
f(\mathbf{X}) &= -\sum_{i=1}^n \left[ \mathbf{a}_i^\top \mathbf{x}_{b_i} - \log \left( \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \right] \\
&= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right]
\end{aligned}
$$

__(b)__ (3 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

**Part 1: Convexity of the Log-Sum-Exp function**

Let $g(\mathbf{u}) = \log\left(\sum_{k=1}^C e^{u_k}\right)$. To prove $g$ is convex, we must show that for any $\mathbf{x}, \mathbf{y} \in \mathbb{R}^C$ and $\theta \in [0, 1]$:
$$g(\theta \mathbf{x} + (1-\theta)\mathbf{y}) \le \theta g(\mathbf{x}) + (1-\theta)g(\mathbf{y})$$

First, we evaluate the term inside the logarithm for the convex combination:
$$\sum_{k=1}^C e^{\theta x_k + (1-\theta)y_k} = \sum_{k=1}^C (e^{x_k})^\theta (e^{y_k})^{1-\theta}$$

We multiply and divide by $\sum_{j=1}^C e^{y_j}$ to isolate a probability distribution:
$$= \left( \sum_{j=1}^C e^{y_j} \right) \sum_{k=1}^C \frac{e^{y_k}}{\sum_{j=1}^C e^{y_j}} \left( \frac{e^{x_k}}{e^{y_k}} \right)^\theta$$

Let $p_k = \frac{e^{y_k}}{\sum_{j=1}^C e^{y_j}}$. Since $p_k \ge 0$ and $\sum_{k=1}^C p_k = 1$, $p_k$ is a valid probability mass function. We can rewrite the sum as an expectation:
$$= \left( \sum_{j=1}^C e^{y_j} \right) \mathbb{E}\left[ \left( \frac{e^{x_K}}{e^{y_K}} \right)^\theta \right]$$

The function $h(t) = t^\theta$ is concave for $\theta \in [0, 1]$ and $t > 0$. By Jensen's inequality for concave functions, $\mathbb{E}[Z^\theta] \le (\mathbb{E}[Z])^\theta$. Applying this yields:
$$ \sum_{k=1}^C p_k \left( \frac{e^{x_k}}{e^{y_k}} \right)^\theta \le \left( \sum_{k=1}^C p_k \frac{e^{x_k}}{e^{y_k}} \right)^\theta = \left( \sum_{k=1}^C \frac{e^{y_k}}{\sum_{j=1}^C e^{y_j}} \frac{e^{x_k}}{e^{y_k}} \right)^\theta = \frac{\left(\sum_{k=1}^C e^{x_k}\right)^\theta}{\left(\sum_{j=1}^C e^{y_j}\right)^\theta} $$

Substituting this bound back into our expanded expression:
$$ \sum_{k=1}^C e^{\theta x_k + (1-\theta)y_k} \le \left( \sum_{j=1}^C e^{y_j} \right) \frac{\left(\sum_{k=1}^C e^{x_k}\right)^\theta}{\left(\sum_{j=1}^C e^{y_j}\right)^\theta} = \left(\sum_{k=1}^C e^{x_k}\right)^\theta \left(\sum_{k=1}^C e^{y_k}\right)^{1-\theta} $$

Taking the natural logarithm of both sides (which preserves the inequality since $\log$ is monotonically increasing) gives:
$$ \log\left(\sum_{k=1}^C e^{\theta x_k + (1-\theta)y_k}\right) \le \theta \log\left(\sum_{k=1}^C e^{x_k}\right) + (1-\theta) \log\left(\sum_{k=1}^C e^{y_k}\right) $$
This matches the definition of convexity, proving $g(\mathbf{u})$ is convex.

---

**Part 2: Convexity of $f(\mathbf{X})$**

Recall the negative log-likelihood function from part (a):
$$ f(\mathbf{X}) = \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right] $$

We can analyze the terms inside the sum:
1. **Linear Term:** The function $\mathbf{X} \mapsto -\mathbf{a}_i^\top \mathbf{x}_{b_i}$ is a linear (affine) operation on the elements of $\mathbf{X}$. All affine functions are convex.
2. **Log-Sum-Exp Term:** The term $\log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)$ is the composition of the log-sum-exp function $g(\mathbf{u})$ and an affine mapping $\mathbf{u}_k = \mathbf{a}_i^\top \mathbf{x}_k$. Since the composition of a convex function with an affine mapping is always convex, this term is convex.

Since $f(\mathbf{X})$ is formed by the sum of convex functions, $f(\mathbf{X})$ itself must be convex.


Since the negative log-likelihood $f$ is convex, every local minimizer is a global minimizer. The maximum likelihood estimation problem is
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

But does $f$ always attain its infimum? The following three questions examine this issue.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


**Difference between Infima and Minima:**

*   **Infimum (inf):** The infimum is the *greatest lower bound* of a function's values. It is the largest constant $P$ such that $f(x) \ge P$ for all $x$ in the domain. 
*   **Minimum (min):** For $P$ to be a minimum, it must be a lower bound ($f(x) \ge P$ for all $x$), and there must exist at least one specific point $x^*$ in the domain such that $f(x^*) = P$. All minima are infima, but not all infima are minima.

**Example of a convex function that does not attain its infimum:**

Consider the exponential function defined on the real numbers, $f: \mathbb{R} \to \mathbb{R}$ given by $f(x) = e^x$. 
*   This function is strictly convex on $\mathbb{R}$.
*   The infimum of this function is $0$, because $e^x > 0$ for all $x \in \mathbb{R}$, and $\lim_{x \to -\infty} e^x = 0$. 
*   However, because there is no finite real number $x$ such that $e^x = 0$, the function never actually achieves this value. Therefore, it has no minimum.


__(d)__ (2 points) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


*Write your answer here.*


Complete separation allows perfect classification of the training data. However, as you will show next, it prevents the existence of a finite maximum likelihood estimate.


__(e)__ (3 points) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


*Write your answer here.*


We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (2 points) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X},
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $\mathbf{a}_i(\mathbf{p}_i - \mathbf{e}_{b_i})^\top\in\mathbb{R}^{p\times C}$ denotes the outer product.



*Write your answer here.*


__(g)__ (4 points) Use row-wise vectorization of $\mathbf{X}$, obtained by stacking its rows one after another into a vector in $\mathbb{R}^{pC}$. Show that the Hessian of $f_\mu$ in these coordinates can be written as
<a id="eq-softmax-hessian"></a>

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}
$$
where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.


*Write your answer here.*


__(h)__ (2 points) Show that $f_\mu$ is $\mu$-strongly convex.


*Write your answer here.*


__(i)__ (2 points) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).


*Write your answer here.*


We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by $1/2$, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(4 points for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.


*Write your answer here.*


__(j-2)__ Using [(1)](#eq-softmax-hessian), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

*Write your answer here.*


__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


*Write your answer here.*


__(l)__ (3 points) KL divergence and NLL. Let $q_i=(q(c\mid\mathbf{a}_i))_{c=1}^C$ be the true conditional label distribution and $\mathbf{p}_i$ the model softmax distribution. Write $\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)$ and show that minimizing its average over the samples is equivalent to minimizing the expected average negative log-likelihood, where the expectation is over labels drawn from $q$ with the features held fixed. Explain how $f(\mathbf{X})/n$ from (a) estimates this expected loss.

*Write your answer here.*


The unregularized maximum likelihood problem may have no finite minimizer. Adding a quadratic regularizer with $\mu>0$ gives a unique regularized estimate, defined by the smooth, strongly convex problem:
<a id="eq-regularized-estimator"></a>

$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{2}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (2 points) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


*Write your answer here.*
